# 02 · Pipeline 1 - CNN 1D para comentarios cualitativos
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Valeria Rudas Ruiz *(Team Leader)*  
> **Objetivo:** Entrenar la CNN 1D con embeddings sobre comentarios de estudiantes.  
> Captura patrones locales (bigramas, trigramas) asociados a cada nivel de desempeño.

---

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

from src.preprocessing import preprocess
from src.models import build_cnn_model, train_cnn, get_cnn_representation, evaluate_model
from src.config import CNN_EMBED_DIM, CNN_FILTERS, CNN_KERNELS, CNN_DROPOUT, CNN_MAX_LEN

UR_RED = "#DA0921"; UR_NAVY = "#242839"; UR_TECH = "#0E6A8C"; UR_GREEN = "#1A6E3A"
print(f"TensorFlow {tf.__version__} ✅")

In [ ]:
data = preprocess("../data/evaluaciones_docentes.csv")
print(f"X_text_train: {data.X_text_train.shape}")
print(f"X_text_test:  {data.X_text_test.shape}")

## 1 · Arquitectura de la CNN 1D

In [ ]:
model = build_cnn_model(num_classes=3)
model.summary()

print("Decisiones de diseno:")
print(f"  Embedding dim={CNN_EMBED_DIM}: entrenable desde cero")
print(f"  Kernels={CNN_KERNELS}: bigramas, trigramas y 4-gramas en paralelo")
print(f"  Filtros={CNN_FILTERS} por rama -> concat={CNN_FILTERS * len(CNN_KERNELS)}d")
print(f"  Dropout={CNN_DROPOUT}: regularizacion principal")
print("  GlobalMaxPooling1D: invariante a la posicion del n-grama")

## 2 · Entrenamiento con EarlyStopping y ReduceLROnPlateau

In [ ]:
cnn_model, cnn_history = train_cnn(
    X_train=data.X_text_train,
    y_train=data.y_train,
    X_val=data.X_text_test,
    y_val=data.y_test,
)
print(f"Épocas entrenadas: {len(cnn_history['accuracy'])}")
print(f"Mejor val_accuracy: {max(cnn_history.get('val_accuracy', [0])):.4f}")

## 3 · Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("loss", "val_loss")],
    ["Accuracy", "Loss"],
):
    train_m, val_m = metric
    epochs = range(1, len(cnn_history[train_m]) + 1)
    ax.plot(epochs, cnn_history[train_m], color=UR_TECH, linewidth=2, label="Train")
    if val_m in cnn_history:
        ax.plot(epochs, cnn_history[val_m], color=UR_RED, linewidth=2,
                linestyle="--", label="Validación")
    ax.set_title(f"CNN 1D - {title}", fontsize=12, fontweight="bold", color=UR_NAVY)
    ax.set_xlabel("Época"); ax.legend(frameon=False)

plt.suptitle("Curvas de aprendizaje CNN 1D", fontsize=14, fontweight="bold", color=UR_RED, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/cnn_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📌 EarlyStopping + ReduceLROnPlateau controlan el overfitting")

## 4 · Evaluación en test

In [ ]:
cnn_proba_test = cnn_model.predict(data.X_text_test, verbose=0)
cnn_pred_test  = np.argmax(cnn_proba_test, axis=1)

metrics_cnn = evaluate_model(
    data.y_test, cnn_pred_test, cnn_proba_test, model_name="CNN 1D"
)

print(f"Accuracy:      {metrics_cnn['accuracy']:.4f}")
print(f"F1-macro:      {metrics_cnn['f1_macro']:.4f}")
print(f"AUC-ROC macro: {metrics_cnn['auc_roc_macro']:.4f}")
print("\nF1 por clase:")
for cls, f1 in metrics_cnn["f1_per_class"].items():
    print(f"  {cls:<12}: {f1:.4f}")
print("\n📌 La CNN sola tiene rendimiento bajo - los comentarios necesitan el complemento numérico del RF")
print("📌 Esto se confirma en el ablation study (04_fusion_ablation.ipynb)")

## 5 · Vector de representación para la fusión

In [ ]:
repr_train = get_cnn_representation(cnn_model, data.X_text_train)
repr_test  = get_cnn_representation(cnn_model, data.X_text_test)

print(f"Vector de representación CNN - train: {repr_train.shape}")
print(f"Vector de representación CNN - test:  {repr_test.shape}")
print(f"\nEstadísticas del vector (post relu - valores >= 0):")
print(f"  min: {repr_train.min():.4f}")
print(f"  max: {repr_train.max():.4f}")
print(f"  mean: {repr_train.mean():.4f}")
print("\n✅ Vector listo para concatenar con probabilidades RF en 04_fusion_ablation.ipynb")

In [ ]:
# Guardar modelo
cnn_model.save("../models/cnn_model.keras")
print("✅ Modelo CNN guardado en models/cnn_model.keras")